# PyHMMER

![PyHMMER](https://proto-bio.github.io/proto-assets/images/tool/pyhmmer/hero.png)

This notebook demonstrates the five PyHMMER tools. `run_pyhmmer_hmmsearch` searches HMM profiles against sequences to find family members. `run_pyhmmer_hmmscan` scans sequences against a profile database to annotate domain content. `run_pyhmmer_phmmer` performs protein-versus-protein search by building a temporary HMM on the fly. `run_pyhmmer_nhmmer` does the same for nucleotide sequences. `run_pyhmmer_jackhmmer` extends phmmer with iterative refinement to detect very distant homologs.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("pyhmmer")
display_overview("pyhmmer")
display_docs_section("pyhmmer", "Background")

## Available tools

In [ ]:
display_available_tools("pyhmmer")

### `run_pyhmmer_hmmsearch`

`run_pyhmmer_hmmsearch` takes one or more HMM profiles and searches them against a set of protein sequences to identify family members. This is the standard approach when you already have a curated profile for a protein family and want to scan a proteome or custom sequence set for matches. Results include per-sequence and per-domain hits with E-values and bit scores.

In [ ]:
from pathlib import Path

import proto_tools
from proto_tools import (
    PyHmmsearchInput,
    PyHmmsearchConfig,
    run_pyhmmer_hmmsearch,
)

In [ ]:
# Display input docs
display_api_reference("pyhmmer", "input", "run_pyhmmer_hmmsearch")

# Path to HMM file containing domain profiles (Pkinase, Barwin, IPK, 1-cysPrx_C)
_repo_root = Path(proto_tools.__file__).resolve().parent.parent
hmm_path = _repo_root / "tests" / "dummy_data" / "test_multiple_hmm.hmm"

# Target proteins: an inositol polyphosphate kinase (matches the IPK + Barwin profiles)
# and hemoglobin alpha (an unrelated negative control)
kinase_seq = (
    "MLEPLPCWDAAKDLKEPQCPPGDRVGVQPGNSRVWQGTMEKAGLAWTRGTGVQSEGTWES"
    "QRQDSDALPSPELLPQDQDKPFLRKACSPSNIPAVIITDMGTQEDGALEETQGSPRGNLP"
    "LRKLSSSSASSTGFSSSYEDSEEDISSDPERTLDPNSAFLHTLDQQKPRVSKSWRKIKNM"
    "VHWSPFVMSFKKKYPWIQLAGHAGSFKAAANGRILKKHCESEQRCLDRLMVDVLRPFVPA"
    "YHGDVVKDGERYNQMDDLLADFDSPCVMDCKMGIRTYLEEELTKARRKPSLRKDMYQKMI"
    "EVDPEAPTEEEKAQRAVTKPRYMQWRETISSTATLGFRIEGIKKEDGTVNRDFKKTKTRE"
    "QVTEAFREFTKGNHNILIAYRDRLKAIRTTLEVSPFFKCHEVIGSSLLFIHDKKEQAKVW"
    "MIDFGKTTPLPEGQTLQHDVPWQEGNREDGYLSGLNNLVDILTEMSQDAPLA"
)

hemoglobin_seq = (
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
    "GSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKL"
    "LSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR"
)

inputs = PyHmmsearchInput(
    hmm=str(hmm_path),
    sequences=[kinase_seq, hemoglobin_seq],
)

In [ ]:
# Display config docs
display_api_reference("pyhmmer", "config", "run_pyhmmer_hmmsearch")

# Report all hits below E-value 10.0 | see docs above for all fields
config = PyHmmsearchConfig(evalue_threshold=10.0)

In [ ]:
# Run HMM profile search
hmmsearch_result = run_pyhmmer_hmmsearch(inputs, config)

In [ ]:
# Display output docs
display_api_reference("pyhmmer", "output", "run_pyhmmer_hmmsearch")

# The kinase (seq 0) should match IPK and Barwin profiles; hemoglobin (seq 1) should not
print(f"Found {hmmsearch_result.num_sequence_hits} sequence hits")
print(f"Found {hmmsearch_result.num_domain_hits} domain hits")

if hmmsearch_result.sequence_hits:
    print("\nSequence hits (target 0=kinase, 1=hemoglobin):")
    for hit in hmmsearch_result.sequence_hits:
        print(f"  query={hit.query_name}, target={hit.target_name}, evalue={hit.evalue:.2e}, score={hit.score:.1f}")

### `run_pyhmmer_hmmscan`

`run_pyhmmer_hmmscan` is the reverse of hmmsearch: given query sequences, it scans them against a database of HMM profiles to annotate which domains are present. This is the standard workflow for domain architecture analysis, answering "what domains does this protein contain?" Results include per-sequence and per-domain hits with conditional E-values relative to the full profile database.

In [ ]:
from proto_tools import (
    PyHmmscanInput,
    PyHmmscanConfig,
    run_pyhmmer_hmmscan,
)

In [ ]:
# Display input docs
display_api_reference("pyhmmer", "input", "run_pyhmmer_hmmscan")

# Scan the same kinase and hemoglobin sequences against the HMM database
inputs = PyHmmscanInput(
    hmm_db=str(hmm_path),
    sequences=[kinase_seq, hemoglobin_seq],
)

In [ ]:
# Display config docs
display_api_reference("pyhmmer", "config", "run_pyhmmer_hmmscan")

# Report all hits below E-value 10.0 | see docs above for all fields
config = PyHmmscanConfig(evalue_threshold=10.0)

In [ ]:
# Run domain scan
hmmscan_result = run_pyhmmer_hmmscan(inputs, config)

In [ ]:
# Display output docs
display_api_reference("pyhmmer", "output", "run_pyhmmer_hmmscan")

# Kinase (seq 0) should be annotated with IPK and Barwin domains
print(f"Found {hmmscan_result.num_sequence_hits} sequence hits")
print(f"Found {hmmscan_result.num_domain_hits} domain hits")

if hmmscan_result.domain_hits:
    print("\nDomain annotations (sequence 0=kinase, 1=hemoglobin):")
    for hit in hmmscan_result.domain_hits:
        print(f"  query={hit.query_name}, domain={hit.target_name}, c_evalue={hit.c_evalue:.2e}, score={hit.domain_score:.1f}")

### `run_pyhmmer_phmmer`

`run_pyhmmer_phmmer` performs protein-versus-protein search by building a temporary HMM profile on the fly from the query sequence and scanning it against target sequences. This gives greater sensitivity to distant homologs than a pairwise alignment like BLASTP without requiring a prebuilt profile. It is well-suited for finding orthologs across species or confirming that a designed sequence falls within a known protein family.

In [ ]:
from proto_tools import (
    PyPhmmerInput,
    PyPhmmerConfig,
    run_pyhmmer_phmmer,
)

In [ ]:
# Display input docs
display_api_reference("pyhmmer", "input", "run_pyhmmer_phmmer")

# Query: human hemoglobin alpha (82 residues)
query_seq = (
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
    "GSAQVKGHGKKVADALTNAVAHVDDMPNALSA"
)

# Three targets at different evolutionary distances
target_seqs = [
    # Mouse hemoglobin alpha: close ortholog, expect strong match
    "MVLSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHFDVSH"
    "GSAQVKGHGKKVADALASAAGHLDDLPGALSAL",
    # Human hemoglobin beta: same globin family, expect partial match
    "VHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST"
    "PDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGT",
    # GFP fragment: unrelated protein, expect no match
    "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTT"
    "GKLPVPWPTLVTTLTYGVQCFSRYPDHMKQ",
]

phmmer_inputs = PyPhmmerInput(
    sequences=[query_seq],
    target_sequences=target_seqs,
)

In [ ]:
# Display config docs
display_api_reference("pyhmmer", "config", "run_pyhmmer_phmmer")

# Report all hits below E-value 10.0 | see docs above for all fields
phmmer_config = PyPhmmerConfig(evalue_threshold=10.0)

In [ ]:
# Run phmmer search
phmmer_result = run_pyhmmer_phmmer(phmmer_inputs, phmmer_config)

In [ ]:
# Display output docs
display_api_reference("pyhmmer", "output", "run_pyhmmer_phmmer")

# Mouse alpha (0) and human beta (1) should hit; GFP (2) should not
print(f"Found {phmmer_result.num_sequence_hits} sequence hits")
print(f"Found {phmmer_result.num_domain_hits} domain hits")

if phmmer_result.sequence_hits:
    print("\nSequence hits (target 0=mouse alpha, 1=human beta, 2=GFP):")
    for hit in phmmer_result.sequence_hits:
        print(f"  query={hit.query_name}, target={hit.target_name}, evalue={hit.evalue:.2e}, score={hit.score:.1f}")

### `run_pyhmmer_nhmmer`

`run_pyhmmer_nhmmer` is the nucleotide counterpart of phmmer. It builds a temporary HMM profile from a DNA or RNA query and scans target nucleotide sequences for homologous regions. This is useful for finding conserved non-coding elements, regulatory regions, or homologous genes at the DNA level where protein-level search is not applicable.

In [ ]:
from proto_tools import (
    PyNhmmerInput,
    PyNhmmerConfig,
    run_pyhmmer_nhmmer,
)

In [ ]:
# Display input docs
display_api_reference("pyhmmer", "input", "run_pyhmmer_nhmmer")

# Query: human hemoglobin alpha coding DNA (first 150 nt)
query_dna = (
    "ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGGGTAAG"
    "GTCGGCGCGCACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTC"
    "CTGTCCTTCCCCACCACCAAGACCTACTTCCCGCACTTCGACCTGAGCCAC"
)

# Two targets: mouse hemoglobin alpha DNA (ortholog) and GFP DNA (unrelated)
target_dna = [
    "ATGGTGCTGTCTGAGGACAAGAGCAACATCAAGGCCGCCTGGGGTAAGATC"
    "GGCGGCCACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTCGCC"
    "TCCTTCCCCACCACCAAGACCTACTTCCCGCACTTCGACGTGAGCCAC",
    "ATGAGTAAAGGAGAAGAACTTTTCACTGGAGTTGTCCCAATTCTTGTTGAAT"
    "TAGATGGTGATGTTAATGGGCACAAATTTTCTGTCAGTGGAGAGGGTGAAGG"
    "TGATGCTACATACGGAAAGCTTACCCTTAAATTTATTTGCACTACTGGAAAA",
]

nhmmer_inputs = PyNhmmerInput(
    sequences=[query_dna],
    target_sequences=target_dna,
)

In [ ]:
# Display config docs
display_api_reference("pyhmmer", "config", "run_pyhmmer_nhmmer")

# Report all hits below E-value 10.0 | see docs above for all fields
nhmmer_config = PyNhmmerConfig(evalue_threshold=10.0)

In [ ]:
# Run nhmmer search
nhmmer_result = run_pyhmmer_nhmmer(nhmmer_inputs, nhmmer_config)

In [ ]:
# Display output docs
display_api_reference("pyhmmer", "output", "run_pyhmmer_nhmmer")

# Mouse alpha DNA (0) should hit; GFP DNA (1) should not
print(f"Found {nhmmer_result.num_sequence_hits} sequence hits")
print(f"Found {nhmmer_result.num_domain_hits} domain hits")

if nhmmer_result.sequence_hits:
    print("\nSequence hits (target 0=mouse alpha DNA, 1=GFP DNA):")
    for hit in nhmmer_result.sequence_hits:
        print(f"  query={hit.query_name}, target={hit.target_name}, evalue={hit.evalue:.2e}, score={hit.score:.1f}")

### `run_pyhmmer_jackhmmer`

`run_pyhmmer_jackhmmer` performs iterative protein sequence search. In the first round it works like phmmer, building a profile from the query and finding initial hits. It then incorporates those hits into a refined profile and searches again, repeating until convergence or the maximum number of iterations. This iterative refinement allows jackhmmer to detect very distant homologs that single-pass methods would miss.

In [ ]:
from proto_tools import (
    PyJackhmmerInput,
    PyJackhmmerConfig,
    run_pyhmmer_jackhmmer,
)

In [ ]:
# Display input docs
display_api_reference("pyhmmer", "input", "run_pyhmmer_jackhmmer")

# Reuse the hemoglobin query and targets from the phmmer example
jackhmmer_inputs = PyJackhmmerInput(
    sequences=[query_seq],
    target_sequences=target_seqs,
)

In [ ]:
# Display config docs
display_api_reference("pyhmmer", "config", "run_pyhmmer_jackhmmer")

# Three iterations to refine the profile | see docs above for all fields
jackhmmer_config = PyJackhmmerConfig(
    max_iterations=3,
    evalue_threshold=10.0,
)

In [ ]:
# Run iterative jackhmmer search
jackhmmer_result = run_pyhmmer_jackhmmer(jackhmmer_inputs, jackhmmer_config)

In [ ]:
# Display output docs
display_api_reference("pyhmmer", "output", "run_pyhmmer_jackhmmer")

# Iterative refinement often finds the beta globin hit at much higher score than phmmer
print(f"Found {jackhmmer_result.num_sequence_hits} sequence hits")
print(f"Found {jackhmmer_result.num_domain_hits} domain hits")

if jackhmmer_result.sequence_hits:
    print("\nSequence hits:")
    for hit in jackhmmer_result.sequence_hits:
        print(f"  query={hit.query_name}, target={hit.target_name}, evalue={hit.evalue:.2e}, score={hit.score:.1f}")

## Export Results

Search results from any PyHMMER function can be exported to CSV or JSON. CSV produces one row per hit with alignment statistics; JSON preserves the full nested structure including domain-level hits.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

jackhmmer_result.export(name="pyhmmer_jackhmmer_results", export_path=output_dir, file_format="csv")
print(f"Results exported to {output_dir}")